userCF的实现代码，根据《推荐系统实践》这本书的思路，结合常用的数据科学计算包完成</br>
不用数据科学包的代码参考:https://github.com/Magic-Bubble/RecommendSystemPractice/blob/master/Chapter2/%E5%9F%BA%E4%BA%8E%E7%94%A8%E6%88%B7%E7%9A%84%E5%8D%8F%E5%90%8C%E8%BF%87%E6%BB%A4%E7%AE%97%E6%B3%95.ipynb </br>

### 读取数据以及预处理
数据只用了三列 用户id，物品id，评分

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
header = ['user_id', 'movie_id', 'ratings', 'timestamp']
ratings_df = pd.read_csv('ratings.dat',
                         sep='::',
                         header=None,
                         engine='python',
                         names=header,
                         usecols=[0,1,2])
#usecols 读取的时候选取列数，这里暂时还用不着时间

In [3]:
ratings_df

,user_id,movie_id,ratings
0,1,1193,5
1,1,661,3
2,1,914,3
3,1,3408,4
4,1,2355,5
...,...,...,...
1000204,6040,1091,1
1000205,6040,1094,5
1000206,6040,562,5
1000207,6040,1096,4


评判UserCF算法时，是将某个用户历史观看记录中的某些物品抽出来作为集合A，假装用户没看过集合A的物品</br>
训练完模型后，查看模型给出推荐的物品中，有多少与集合A的物品重合</br>

stratify=ratings_df['user_id'] 这个参数的作用是分割，stratify参数必须加上，因为该数据是有序的，在9/10处直接分割会导致无法评估算法<br>
比如，在9/10处直接画一条线，会导致前9/10的数据是用户ID为5500的，剩下的1/10是ID为5501~6040，如果是这样做，就无法使用刚刚那种方法评测算法强度

In [4]:
ratings_df_train,ratings_df_test = train_test_split(ratings_df,test_size = 0.1,stratify=ratings_df['user_id'])

In [5]:
# 划分后给训练集按照user_id排序(因为之前的划分过程中被打乱了)
ratings_df_train.sort_values(by='user_id', inplace=True, ascending=True)
print(ratings_df_train.shape)
ratings_df_train

(900188, 3)


,user_id,movie_id,ratings
49,1,531,4
6,1,1287,5
32,1,1566,4
1,1,661,3
30,1,2294,4
...,...,...,...
999946,6040,608,4
1000028,6040,1991,1
999887,6040,916,5
1000183,6040,3735,4


In [6]:
# 同样的，给测试集排序
ratings_df_test.sort_values(by='user_id', inplace=True, ascending=True)
print(ratings_df_test.shape)
ratings_df_test

(100021, 3)


,user_id,movie_id,ratings
41,1,1961,5
36,1,1836,5
26,1,1097,4
15,1,2791,4
40,1,1,5
...,...,...,...
1000126,6040,1333,4
1000171,6040,3388,1
1000199,6040,2022,5
1000159,6040,3342,3


### 建立用户-物品索引矩阵

行：用户ID，列：物品ID </br>
交叉值：某用户对某物品的兴趣，如果没看过则是0 </br>
将数据按照'user_id'和'movie_id'进行透视，创建用户-物品矩阵，user_item_matrix作矩阵用于今后推荐 </br>
这个矩阵就是用户->物品的索引，记录着每个用户对电影的感兴趣程度（根据打分来实现，没有打分说明完全没有兴趣） </br>
从这里来看，整个矩阵是比较稀疏的，符合现实情况。 </br>

In [7]:
user_item_matrix = ratings_df_train.pivot(index='user_id', columns='movie_id', values='ratings').fillna(0)
user_item_matrix

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6036,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 建立用户-用户索引
矩阵内数值代表用户的相似程度，越接近1表明两个用户越相似</br>
这里的计算使用了课堂上介绍的第一种方式，直接使用两个用户间喜欢的物品交集，并未使用额外的热度作补正 </br>

In [8]:
# 直接调用consine_similarity计算用户相似度矩阵
user_similarity_matrix = cosine_similarity(user_item_matrix)
user_similarity_matrix

array([[1.        , 0.10891524, 0.10843532, ..., 0.        , 0.15065947,
        0.10604677],
       [0.10891524, 1.        , 0.13974533, ..., 0.05181417, 0.05868522,
        0.19809352],
       [0.10843532, 0.13974533, 1.        , ..., 0.03425881, 0.07866018,
        0.12512297],
       ...,
       [0.        , 0.05181417, 0.03425881, ..., 1.        , 0.17582855,
        0.07769396],
       [0.15065947, 0.05868522, 0.07866018, ..., 0.17582855, 1.        ,
        0.20657793],
       [0.10604677, 0.19809352, 0.12512297, ..., 0.07769396, 0.20657793,
        1.        ]])

### 用户相似度矩阵

行和列都是用户ID
交叉位置是两个用户的相似度

In [9]:
# 将相似度矩阵转换为DataFrame，user_similarity_df作用户相似度矩阵，用于推荐
user_similarity_df = pd.DataFrame(user_similarity_matrix, index=user_item_matrix.index, columns=user_item_matrix.index)
user_similarity_df

user_id,1,2,3,4,5,6,7,8,9,10,...,6031,6032,6033,6034,6035,6036,6037,6038,6039,6040
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.108915,0.108435,0.083610,0.100225,0.157618,0.066724,0.108144,0.168610,0.232865,...,0.145206,0.067148,0.049167,0.036998,0.097253,0.185757,0.102085,0.000000,0.150659,0.106047
2,0.108915,1.000000,0.139745,0.116026,0.114556,0.077376,0.242007,0.227504,0.175573,0.190857,...,0.093011,0.066474,0.188916,0.015753,0.172788,0.193116,0.172346,0.051814,0.058685,0.198094
3,0.108435,0.139745,1.000000,0.168686,0.068653,0.036470,0.151837,0.072872,0.138336,0.201383,...,0.074611,0.070887,0.179767,0.000000,0.096584,0.132340,0.080299,0.034259,0.078660,0.125123
4,0.083610,0.116026,0.168686,1.000000,0.012040,0.000000,0.147008,0.083284,0.076431,0.125761,...,0.153691,0.093931,0.400084,0.000000,0.092963,0.161914,0.096602,0.070221,0.072160,0.139074
5,0.100225,0.114556,0.068653,0.012040,1.000000,0.052469,0.082634,0.207257,0.255936,0.117491,...,0.100758,0.039216,0.059059,0.058740,0.165257,0.267372,0.175112,0.022344,0.030244,0.210204
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6036,0.185757,0.193116,0.132340,0.161914,0.267372,0.097290,0.094816,0.214658,0.223487,0.310515,...,0.120007,0.201191,0.162639,0.106189,0.223258,1.000000,0.300051,0.112203,0.213332,0.359625
6037,0.102085,0.172346,0.080299,0.096602,0.175112,0.065892,0.072903,0.137404,0.195237,0.194002,...,0.123455,0.246895,0.092168,0.117091,0.134350,0.300051,1.000000,0.035533,0.198051,0.378682
6038,0.000000,0.051814,0.034259,0.070221,0.022344,0.027819,0.000000,0.021082,0.088157,0.097427,...,0.075056,0.088296,0.045322,0.000000,0.070541,0.112203,0.035533,1.000000,0.175829,0.077694


以下两种方法都可以看出两个用户的相似度，第一种是通过用户ID，第二种是通过行列号

In [10]:
user_similarity_df.loc[1,6]

np.float64(0.1576182233669603)

In [11]:
user_similarity_df.iloc[0,5]

np.float64(0.1576182233669603)

### 给用户作推荐 
到这里，我们已经计算得出了用户-物品索引，与用户-用户索引，可以开始给用户做推荐了</br>
有两个超参数可以自己设置，重复多次作实验：</br>
k：选取多少个最相似的用户，如在本代码中k=80，则选前80个与用户1最相似的用户</br>
n：每个相似用户选取多少个物品作为打分综合，如果n=3，则每个用户选取3个最相似的物品，在这里没有使用n，而是选取了所有物品</br>
在线上，n的意义在于不将所有的物品枚举出来（一个活跃用户或许已经在平台看了上万个视频/笔记，如果要全部拿出会增大计算量，因此n的意义是限制，这个问题中本身计算量不大，因此不需要做限制）

In [12]:
# target user: 目标用户，也就是给谁推荐，=1意味着给ID为1的用户推荐
target_user = 3

# target user: similar_population_K，在算法中划出前K个与这个人相似的人作推荐，=80意味着划出平台中跟他最相似的前80个人
similar_population_K = 80

# 获取前K位与目标用户相似的用户
target_user_similarity = user_similarity_df.loc[target_user]
similar_users = target_user_similarity.sort_values(ascending=False).head(similar_population_K+1)[1:] # +1是因为去除自己的记录（自己跟自己最相似）
similar_users

user_id
3000    0.352223
479     0.346524
2806    0.333467
3148    0.330722
3730    0.329045
          ...   
2756    0.270360
1308    0.270141
5437    0.269890
5975    0.269807
2487    0.269670
Name: 3, Length: 80, dtype: float64

In [13]:
# 获取目标用户所观看过的电影情况
target_user_movies = user_item_matrix.loc[target_user]
print(target_user_movies)

# 获取这些相似用户观看过的电影
similar_users_movies = user_item_matrix.loc[similar_users.index]
similar_users_movies

movie_id
1       0.0
2       0.0
3       0.0
4       0.0
5       0.0
       ... 
3948    0.0
3949    0.0
3950    0.0
3951    0.0
3952    0.0
Name: 3, Length: 3691, dtype: float64


movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
3000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
479,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0
2806,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0
3148,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2756,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5437,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# 将用户相似度与电影评分相乘，得到加权后的矩阵
weighted_ratings = similar_users_movies.mul(similar_users, axis=0)
weighted_ratings

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
3000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.352223,1.056670,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
479,1.732620,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.732620,0.0,0.0,0.0,0.0
2806,0.666934,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.333867,0.0,0.0,0.0,0.0
3148,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3730,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2756,1.351802,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.351802,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1308,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
5437,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [15]:
# 计算推荐分数，这里简单地将相似用户的评分相加
recommendations = weighted_ratings.sum().sort_values(ascending=False)
recommendations

movie_id
260     95.361417
1198    92.823826
1196    92.532189
1197    85.688976
1210    83.350082
          ...    
1056     0.000000
3926     0.000000
1057     0.000000
3928     0.000000
3929     0.000000
Length: 3691, dtype: float64

In [16]:
# 过滤掉目标用户已经观看过的电影(看过的物品不推荐)
recommendations = recommendations[~target_user_movies.astype(bool)]
recommendations

movie_id
589     55.345775
592     55.213703
2571    53.836240
110     51.813043
2916    49.179449
          ...    
1056     0.000000
3926     0.000000
1057     0.000000
3928     0.000000
3929     0.000000
Length: 3645, dtype: float64

In [17]:
# 取统计分的前10进行推荐
recommendations[:10]

movie_id
589     55.345775
592     55.213703
2571    53.836240
110     51.813043
2916    49.179449
1073    48.097264
1240    47.939269
2918    47.564511
858     47.136437
1036    45.805899
dtype: float64

In [ ]:
# 给用户3的推荐结果，取评分前十的物品ID
recommendation_result = recommendations[:10].index
recommendation_result

Index([589, 592, 2571, 110, 2916, 1073, 1240, 2918, 858, 1036], dtype='int64', name='movie_id')

### 离线评估推荐结果

In [19]:
# 用户1在测试集里评价过的电影（在这个实验中因为被划为测试集，所以当成是他没看过，但喜欢的电影）
test_data_user_1 = ratings_df_test[ratings_df_test['user_id'] == 1]
real_result = test_data_user_1["movie_id"]
real_result

41    1961
36    1836
26    1097
15    2791
40       1
Name: movie_id, dtype: int64

In [20]:
# 取交集用于算指标，即推荐中他喜欢的百分比
intersection_result = recommendation_result.intersection(real_result)
intersection_result

Index([], dtype='int64', name='movie_id')

In [21]:
# 算出精确度
precision = len(intersection_result) / len(recommendation_result)
precision

0.0

### 将上述代码整合成函数，输入需要作推荐的用户ID，用户-物品索引和用户-用户索引，以及超参数，得到推荐结果

In [22]:
# 为了评测代码，将获取推荐的过程写成函数（用户的相似度矩阵不用再浪费时间计算）
# 函数输入用户ID，相似度矩阵，用户-物品矩阵，要找多少个相似的人做统计，推荐前多少部电影，输出为推荐的电影

def get_user_recommendations(target_user, user_similarity_df, user_item_matrix, similar_population_K, top_number=10):

    target_user_similarity = user_similarity_df.loc[target_user]
    similar_users = target_user_similarity.sort_values(ascending=False).head(similar_population_K+1)[1:]

    # 获取目标用户观看电影的一个情况
    target_user_movies = user_item_matrix.loc[target_user]

    # 获取相似用户观看过的电影
    similar_users_movies = user_item_matrix.loc[similar_users.index]

    # 将用户相似度与电影评分相乘，得到加权后的矩阵
    weighted_ratings = similar_users_movies.mul(similar_users, axis=0)

    # 计算推荐分数，这里简单地将相似用户的评分相加，可以根据实际需求调整推荐算法
    recommendations = weighted_ratings.sum().sort_values(ascending=False)

    # 过滤掉目标用户已经观看过的电影
    recommendations = recommendations[~target_user_movies.astype(bool)]

    # 取统计分的前top_number个作推荐
    return recommendations[:top_number]

In [23]:
# 测试函数能否使用
get_user_recommendations(3, user_similarity_df, user_item_matrix, 80, 10).index

Index([589, 592, 2571, 110, 2916, 1073, 1240, 2918, 858, 1036], dtype='int64', name='movie_id')

### 给1~6040所有的用户做推荐，算出总体的离线指标

In [26]:
# 计算全局的精确度
# 初始化变量
similar_population_K = 20
top_number = 10

recommend_hit = 0
real_all = 0
recall_hit = 0
real_all_items = 0

# 遍历每个用户
for target_user in set(ratings_df_test["user_id"]):
    test_data_user = ratings_df_test[ratings_df_test['user_id'] == target_user]
    real_result = test_data_user["movie_id"]

    # 获取推荐结果
    recommendation_result = get_user_recommendations(target_user, user_similarity_df, user_item_matrix, similar_population_K, top_number).index

    # 计算推荐命中数
    intersection_result = recommendation_result.intersection(real_result)
    recommend_hit += len(intersection_result)
    real_all += top_number

    # 计算召回率相关指标
    recall_hit += len(intersection_result)
    real_all_items += len(real_result)

# 计算精确度、召回率和 F1 分数
precision = recommend_hit / real_all
recall = recall_hit / real_all_items
f1_score = 2 * (precision * recall) / (precision + recall)

# 输出结果
print(f"Precision精确度: {precision}")
print(f"Recall召回率: {recall}")
print(f"F1 Score: {f1_score}")

Precision精确度: 0.19827814569536423
Recall召回率: 0.11973485568030713
F1 Score: 0.14930713559945394
